# 用 Sciverse 做结构化论文筛选

> 通过 meta-catalog 获取可用字段，用 meta-search 精确过滤论文

---

**Sciverse Cookbook** | [文档首页](https://sciverse.space/docs#cookbook) | [申请 API Key](https://sciverse.space/docs#auth)

## 前置准备

1. 在 [Sciverse 控制台](https://sciverse.space/) 申请 API Token
2. 安装依赖：`pip install httpx anthropic`
3. 将下方 `sv-...` 替换为你的真实 Token


In [ ]:
# 安装依赖（Colab 环境）
!pip install -q httpx anthropic


## Step 1: 获取可用筛选字段

调用 meta-catalog 查看有哪些结构化字段可用


In [ ]:
import httpx

BASE = "https://api.sciverse.space"
HEADERS = {"Authorization": "Bearer sv-..."}

async def get_catalog():
    async with httpx.AsyncClient() as client:
        resp = await client.get(f"{BASE}/meta-catalog", headers=HEADERS)
        return resp.json()

catalog = await get_catalog()
print("Available fields:")
for field in catalog["fields"]:
    print(f"  - {field['name']} ({field['type']}): {field.get('description', '')}")

## Step 2: 构建结构化筛选查询

使用 meta-search 按年份、期刊、作者等条件过滤


In [ ]:
async def meta_search(query: str, filters: list, top_k: int = 20):
    async with httpx.AsyncClient() as client:
        resp = await client.post(
            f"{BASE}/meta-search",
            headers=HEADERS,
            json={"query": query, "filters": filters, "top_k": top_k}
        )
        return resp.json()

# 筛选 2022-2024 年 Nature/Science 上的 CRISPR 论文
results = await meta_search(
    query="CRISPR base editing therapeutic applications",
    filters=[
        {"field": "year", "op": "gte", "value": 2022},
        {"field": "journal", "op": "in", "value": ["Nature", "Science", "Cell"]},
    ],
    top_k=15
)
print(f"Found {len(results['hits'])} papers matching criteria")
for h in results["hits"][:5]:
    print(f"  [{h['year']}] {h['title']}")

---

## 下一步

- 📖 [查看完整 API 文档](https://sciverse.space/docs#sciverse/api)
- 🔑 [申请 / 管理 API Token](https://sciverse.space/)
- 📚 [浏览更多 Cookbook 案例](https://sciverse.space/docs#cookbook)
- 💬 [加入开发者社群](https://sciverse.space/)
